# From Coefficient to Recommendation

**DS4DH Practice Pack · Module 11 — Policy and Domain Reasoning**

*Technique:* Reading a confidence interval as a policy constraint

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/11b_coefficient_to_policy.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

A point estimate is what a model thinks is most likely. A **confidence interval**
is the range it cannot rule out — and for a policy decision the interval is
usually the more important object, because programmes have to be sized for the
plausible range rather than the single best guess.

This notebook takes a regression coefficient and works it all the way to a
budgeted recommendation, showing where each assumption enters.

In [ ]:
csd = df.dropna(subset=['csd_code'])

reg_df = csd[csd['immigrant_status'].isin(['Immigrant', 'Non-immigrants'])
             & csd['cma'].isin(CITIES)].dropna(subset=['Renter']).copy()
reg_df['is_immigrant'] = (reg_df['immigrant_status'] == 'Immigrant').astype(int)

print(f'{len(reg_df)} rows — one per (CSD, immigrant status) with a renter STIR')
print(reg_df['immigrant_status'].value_counts().to_string())

In [ ]:
city_dummies = pd.get_dummies(reg_df['cma'], drop_first=True, dtype=float)
X = sm.add_constant(pd.concat([reg_df[['is_immigrant']], city_dummies], axis=1))
model = sm.OLS(reg_df['Renter'], X).fit(cov_type='HC1')

coef = model.params['is_immigrant']
ci = model.conf_int().loc['is_immigrant']

print(f'coefficient    {coef:+.3f} pp')
print(f'95% CI         [{ci[0]:+.3f}, {ci[1]:+.3f}] pp')
print(f'p-value        {model.pvalues["is_immigrant"]:.4f}')
print()
print('The interval spans zero and both signs. Whatever else this supports, it')
print('does not support a claim that the effect is positive.')

## What an interval spanning zero means for a decision

It does not mean the effect is zero. It means the data cannot distinguish a small
penalty from a small advantage — and a programme designed on the point estimate
alone would be sized for a number the data cannot defend.

In [ ]:
MEDIAN_RENT_INCOME = reg_df['rent_income'].median()
N_HOUSEHOLDS = 10000     # a hypothetical programme's target population

print(f'Median renter household income : ${MEDIAN_RENT_INCOME:,.0f}')
print(f'Hypothetical programme size    : {N_HOUSEHOLDS:,} households')
print()
print(f'{"scenario":<22}{"pp":>8}{"per household/yr":>20}{"programme cost":>18}')
print('-' * 68)
for label, pp in [('CI lower bound', ci[0]), ('point estimate', coef), ('CI upper bound', ci[1])]:
    per = MEDIAN_RENT_INCOME * pp / 100
    print(f'{label:<22}{pp:>+8.2f}{per:>20,.0f}{per * N_HOUSEHOLDS:>18,.0f}')
print()
print('The interval implies a programme cost that could be negative. That is')
print('the honest answer: this analysis does not justify a programme.')

## The finding that does support something

Module 04's Edmonton result had a CI that excludes zero. Run the same pipeline on
it and the arithmetic becomes usable.

In [ ]:
from scipy import stats

csd = df.dropna(subset=['csd_code'])
imm = csd[csd['immigrant_status'] == 'Immigrant'][['csd_code', 'cma', 'Renter', 'rent_income']]
nim = csd[csd['immigrant_status'] == 'Non-immigrants'][['csd_code', 'Renter']]
pen = imm.merge(nim, on='csd_code', suffixes=('_imm', '_nim')).dropna(
    subset=['Renter_imm', 'Renter_nim'])
ed = pen[pen['cma'] == 'Edmonton']
gap = ed['Renter_imm'] - ed['Renter_nim']

ed_ci = stats.t.interval(0.95, len(gap) - 1, loc=gap.mean(), scale=stats.sem(gap))
print(f'Edmonton, n = {len(gap)} CSDs')
print(f'  mean gap  {gap.mean():+.2f} pp')
print(f'  95% CI    [{ed_ci[0]:+.2f}, {ed_ci[1]:+.2f}] pp')
print()
print('This interval excludes zero, so the direction is defensible. Note it is')
print('still wide — a factor of ~5 between the bounds.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
items = [('Pooled 4-city\n(city FE, HC1)', coef, ci[0], ci[1]),
         ('Edmonton\n(paired CSDs)', gap.mean(), ed_ci[0], ed_ci[1])]
for i, (label, c, lo, hi) in enumerate(items):
    ax.plot([lo, hi], [i, i], lw=3, color='#3DA5D9')
    ax.plot(c, i, 'o', ms=9, color='#E8663D')
ax.axvline(0, color='#333', ls='--', lw=1.5)
ax.set_yticks(range(len(items)))
ax.set_yticklabels([t[0] for t in items])
ax.set_xlabel('immigrant − non-immigrant renter STIR (pp)')
ax.set_title('One interval crosses zero; one does not')
plt.tight_layout()
plt.show()

### 🔧 Your turn 1

Change `N_HOUSEHOLDS` to 50,000 and re-run the costing table.

The uncertainty scales with the programme. At what programme size does the width
of the interval become larger than the amounts most departments can absorb as
error?

### 🔧 Your turn 2

Recompute the Edmonton interval at 99% confidence
(`stats.t.interval(0.99, ...)`).

Does it still exclude zero? Which confidence level belongs in a policy brief, and
who should be making that choice — the analyst or the department?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Cost uncertainty scales linearly with programme size, so at
50,000 households the interval spans tens of millions. This is the practical
argument for reporting intervals rather than point estimates: at small scale the
uncertainty is absorbable and at large scale it is the dominant budget risk, and
a point estimate makes those two situations look identical.

**Your turn 2.** At 99% the Edmonton interval usually still excludes zero, though
narrowly. The confidence level is a decision about how much risk of a false
positive the department is willing to carry, and that is *their* call, not the
analyst's — the analyst's job is to present the interval at more than one level
and explain what each implies. Picking 95% silently because it is conventional
hides a decision that belongs to the reader.

</details>

## Where this stops

Module 11's conclusion for this dataset is uncomfortable and correct: the pooled
analysis supports no programme, and the one robust finding supports a targeting
adjustment rather than a spend.

Next: the capstone, which asks you to say all of this in one page.